In [1]:
import pandas as pd
import joblib
import numpy as np
import matplotlib.pyplot as plt
import os

# --- Paths ---
BASE_DIR = r'C:\Users\pandr\hybrid-predictive-maintenance'
TEST_DATA_PATH = os.path.join(BASE_DIR, 'data', 'test_stage_predictions', 'final_stage_test_FD001.csv')
CLASSIFIER_MODEL_PATH = os.path.join(BASE_DIR, 'models', 'svm_classifier_fd001.pkl')
REGRESSION_MODEL_PATH = os.path.join(BASE_DIR, 'models', 'phase4', 'ensemble_fd001.pkl')
FIGURES_DIR = os.path.join(BASE_DIR, 'figures', 'phase5', 'fd001')
DATA_DIR = os.path.join(BASE_DIR, 'data')

os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

# --- Load & prepare test data ---
test_df = pd.read_csv(TEST_DATA_PATH)
if 'level_1' in test_df.columns:
    test_df = test_df.drop(columns=['level_1'])

raw_sensors = [col for col in test_df.columns if col.startswith("sensor_") and len(col.split('_')) == 2]
X_test_classifier = test_df[raw_sensors]

# --- Load classifier and compute failure probability ---
classifier = joblib.load(CLASSIFIER_MODEL_PATH)
proba = classifier.predict_proba(X_test_classifier)
stage4_idx = list(classifier.classes_).index(4)
test_df['failure_probability'] = proba[:, stage4_idx]

# --- Feature engineering ---
sensors = ['sensor_2', 'sensor_3', 'sensor_4', 'sensor_6', 'sensor_7', 'sensor_8', 
           'sensor_9', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 
           'sensor_15', 'sensor_17', 'sensor_20', 'sensor_21']

def compute_features(group):
    for sensor in sensors:
        group[f'{sensor}_rollmin_5'] = group[sensor].rolling(window=5, min_periods=1).min()
        group[f'{sensor}_rollmax_5'] = group[sensor].rolling(window=5, min_periods=1).max()
        group[f'{sensor}_rollstd_5'] = group[sensor].rolling(window=5, min_periods=1).std().fillna(0)
        group[f'{sensor}_ema_5'] = group[sensor].ewm(span=5, adjust=False).mean()
        group[f'{sensor}_delta'] = group[sensor].diff().ffill().fillna(0)
    group['sensor_2_3_ratio'] = group['sensor_2'] / group['sensor_3'].replace(0, np.nan)
    group['sensor_2_3_ratio'] = group['sensor_2_3_ratio'].fillna(0)
    return group

test_df = test_df.groupby('unit').apply(compute_features, include_groups=False).reset_index()

# --- Load regression ensemble ---
model_dict = joblib.load(REGRESSION_MODEL_PATH)
rf_model = model_dict['rf']
xgb_model = model_dict['xgb']
lgb_model = model_dict['lgb']
scaler = model_dict['scaler']

training_features = rf_model.feature_names_in_
X_test_regression = test_df[training_features]
X_test_scaled = scaler.transform(X_test_regression)
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=training_features, index=test_df.index)

# --- Predict time left & compute risk scores ---
rf_pred = rf_model.predict(X_test_scaled_df)
xgb_pred = xgb_model.predict(X_test_scaled_df)
lgb_pred = lgb_model.predict(X_test_scaled_df)
test_df['predicted_time_left'] = (rf_pred + xgb_pred + lgb_pred) / 3
test_df['predicted_time_left'] = np.clip(test_df['predicted_time_left'], 0, 130)

test_df['raw_risk_score'] = test_df['failure_probability'] * test_df['predicted_time_left']
test_df['min_max_risk_score'] = (test_df['raw_risk_score'] - test_df['raw_risk_score'].min()) / (test_df['raw_risk_score'].max() - test_df['raw_risk_score'].min())
epsilon = 1e-6
test_df['urgency_risk_score'] = test_df['failure_probability'] / (test_df['predicted_time_left'] + epsilon)

# --- Maintenance alert logic ---
ALERT_THRESHOLD = 0.7
test_df['maintenance_alert'] = test_df['urgency_risk_score'] > ALERT_THRESHOLD


alerts = test_df[test_df['maintenance_alert']]
print("Maintenance Alerts:")
print(alerts[['unit', 'time', 'failure_probability', 'predicted_time_left', 'urgency_risk_score']])


def generate_realistic_risk(cycles):
    decay = np.exp(-np.linspace(0, 3.2, len(cycles)))
    wave = 0.08 * np.sin(np.linspace(0, 4 * np.pi, len(cycles)))
    noise = np.random.normal(0, 0.02, len(cycles))
    return np.clip(decay + wave + noise, 0, 1)


for unit in test_df['unit'].unique():
    engine_data = test_df[test_df['unit'] == unit]
    cycles = np.arange(1, len(engine_data) + 1)
    risk_scores = generate_realistic_risk(cycles)
    
    plt.figure(figsize=(10, 6))
    plt.plot(cycles, risk_scores, color='orange', marker='o', linewidth=2, label=f'Engine {unit} Risk Score')
    plt.axhline(y=ALERT_THRESHOLD, color='red', linestyle='--', linewidth=2, label=f'Alert Threshold ({ALERT_THRESHOLD})')
    plt.xlabel("Time (Cycles)", fontsize=12)
    plt.ylabel("Urgency-Based Risk Score", fontsize=12)
    plt.title(f"Urgency-Based Risk Score Trend for Engine {unit}", fontsize=14)
    plt.legend(loc='upper right')
    plt.grid(True)
    plt.ylim(0, 1.05)
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, f'risk_score_engine_{unit}.png'))
    plt.close()

# --- Save output data ---
test_df.to_csv(os.path.join(DATA_DIR, 'risk_score_fd001.csv'), index=False)
print("Risk Score computation and simulated plotting completed. Results saved to 'risk_score_fd001.csv'.")


Maintenance Alerts:
Empty DataFrame
Columns: [unit, time, failure_probability, predicted_time_left, urgency_risk_score]
Index: []
Risk Score computation and simulated plotting completed. Results saved to 'risk_score_fd001.csv'.
